# CodSoft Data Analytics Internship — Task 2
## Exploratory Data Analysis of Retail Sales

This notebook follows the EDA task: descriptive statistics, trends, relationships, outlier detection, business questions, and a short report.

## 1. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
INPUT_FILE = PROJECT_DIR / "data" / "sales_data.csv"
OUTPUT_DIR = PROJECT_DIR / "output"
CHARTS_DIR = PROJECT_DIR / "charts"
OUTPUT_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_FILE, parse_dates=["OrderDate"])
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df.head()

## 2. Inspect the dataset and descriptive statistics

In [ ]:
print("Rows and columns:", df.shape)
df.info()
display(df.describe(include="all"))
display(df.isnull().sum().to_frame("Missing count"))

## 3. Which region generated the highest revenue?

In [ ]:
region = df.groupby("Region", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)
display(region)
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(region["Region"], region["Revenue"])
ax.set(title="Total Revenue by Region", xlabel="Region", ylabel="Revenue")
plt.tight_layout()
plt.show()
region.to_csv(OUTPUT_DIR / "revenue_by_region.csv", index=False)

## 4. Which category generated the highest revenue?

In [ ]:
category = df.groupby("Category", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)
display(category)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(category["Category"], category["Revenue"])
ax.set(title="Total Revenue by Product Category", xlabel="Category", ylabel="Revenue")
plt.tight_layout()
plt.show()
category.to_csv(OUTPUT_DIR / "revenue_by_category.csv", index=False)

## 5. Monthly revenue trend

In [ ]:
df["Month"] = df["OrderDate"].dt.to_period("M").astype(str)
monthly = df.groupby("Month", as_index=False)["Revenue"].sum()
display(monthly)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly["Month"], monthly["Revenue"], marker="o")
ax.set(title="Monthly Revenue Trend", xlabel="Month", ylabel="Revenue")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
monthly.to_csv(OUTPUT_DIR / "monthly_revenue.csv", index=False)

## 6. Distribution and relationship between variables

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["Revenue"], bins=12)
ax.set(title="Distribution of Order Revenue", xlabel="Revenue", ylabel="Number of Orders")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for label, part in df.groupby("Category"):
    ax.scatter(part["Quantity"], part["Revenue"], label=label)
ax.set(title="Order Quantity vs Revenue", xlabel="Quantity", ylabel="Revenue")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Detect potential outliers using IQR

In [ ]:
q1, q3 = df["Revenue"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(df["Revenue"] < lower_bound) | (df["Revenue"] > upper_bound)].copy()
print("Lower bound:", lower_bound, "Upper bound:", upper_bound)
display(outliers[["OrderDate", "Region", "Product", "Revenue"]])
outliers.to_csv(OUTPUT_DIR / "potential_outliers.csv", index=False)

## 8. Summary

In [ ]:
print("Total revenue:", round(df["Revenue"].sum(), 2))
print("Highest-revenue region:")
display(region.head(1))
print("Highest-revenue category:")
display(category.head(1))
print("Potential outlier count:", len(outliers))

## Conclusion
This analysis uses Pandas for summaries and Matplotlib for visualizations, then applies the IQR method to flag potential outliers. The dataset is synthetic and intended for practice.